# Финальная Подготовка Модели (Clean Backtest)

В этом ноутбуке мы обучаем финальную модель, которую готовы отправить на реальную торговлю. Мы жестко отрезаем последние 2 месяца истории. Нейросеть учится на всем, кроме этих двух месяцев. Затем мы прогоняем её через отрезанный кусок и смотрим на результат. Если плюс — модель идет в продакшен.

In [1]:
# %load_ext autoreload
# %autoreload 2

import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize, VecFrameStack
from stable_baselines3.common.monitor import Monitor
import wandb
from wandb.integration.sb3 import WandbCallback


from utils.data_loader import load_crypto_data
from utils.feature_generator import FeatureGenerator
from custom_envs.trading_env_v5 import TradingEnvV5
from agents.ppo_agent import create_ppo_agent, TradingMetricsCallback


# Инициализация Weights & Biases
# sync_tensorboard=True заставит W&B автоматически копировать все графики из TensorBoard в облако
run = wandb.init(
    project="trading_rl",
    name="ppo_cnn_gated_hmm_2M",
    sync_tensorboard=True,  
    monitor_gym=True,       
    save_code=True,
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/maximsinyaev/.netrc.
wandb: Currently logged in as: bemaxradio (bemaxradio-sinyaev) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [agents] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


wandb: WARNING Artifact "source-trading_rl-_Users_maximsinyaev_projects_trading_rl_trading_rl_10._prod_train_and_backtest.ipynb" already exists with the same content. No new version will be created.
wandb: WARNING Artifact "source-trading_rl-_Users_maximsinyaev_projects_trading_rl_trading_rl_10._prod_train_and_backtest.ipynb" already exists with the same content. No new version will be created.
wandb: WARNING Artifact "source-trading_rl-_Users_maximsinyaev_projects_trading_rl_trading_rl_10._prod_train_and_backtest.ipynb" already exists with the same content. No new version will be created.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

print("Настройка путей для сохранения/загрузки моделей:")
model_widget = widgets.Text(value="models/ppo_prod_model", description="Model Path:", layout=widgets.Layout(width='50%'))
norm_widget = widgets.Text(value="models/vec_normalize_prod.pkl", description="Norm Path:", layout=widgets.Layout(width='50%'))
display(model_widget, norm_widget)

## 1. Загрузка Данных и Train/Test Split

In [2]:
df = load_crypto_data(symbol='BTCUSDT', start_date='2020-01-01', interval='4h', source='bybit_futures')

fg = FeatureGenerator()
data_features = fg.transform(df)

hmm_cols = [c for c in data_features.columns if 'hmm_regime' in c]
print(f"Data Features shape: {data_features.shape}")

# Отрезаем последние 2 месяца (6 свечей/день * 60 дней = 360 свечей)
TEST_SIZE = 360

train_df = data_features.iloc[:-TEST_SIZE].reset_index(drop=True)
test_df = data_features.iloc[-TEST_SIZE:].reset_index(drop=True)

print(f"Train size: {len(train_df)} свечей")
print(f"Test size: {len(test_df)} свечей")

📥 Need to download 367 missing dates
  📅 Downloading 2020...


    📊 Downloading funding rates...


  📅 Downloading 2026...


    📊 Downloading funding rates...


    💾 Saved 2020: BTCUSDT_2020.parquet.gz
    💾 Saved 2026: BTCUSDT_2026.parquet.gz
✅ HMM Model loaded from /Users/maximsinyaev/projects/trading_rl/trading_rl/models/hmm_model.pkl
Data Features shape: (13597, 50)
Train size: 13237 свечей
Test size: 360 свечей


## 2. Инициализация и Обучение на Train

In [3]:
N_STACK = 10

train_env = TradingEnvV5(df=train_df, continuous_action=True, t_max=1000)
vec_train = VecFrameStack(DummyVecEnv([lambda: Monitor(train_env)]), n_stack=N_STACK)
vec_train = VecNormalize(vec_train, norm_obs=True, norm_reward=True, clip_obs=10., clip_reward=10.)

model = create_ppo_agent(
    vec_train, 
    extractor_type="cnn", 
    num_hmm_states=len(hmm_cols), 
    n_stack=N_STACK,
    tensorboard_log="./tensorboard_logs_prod/"
)

callbacks = [
    TradingMetricsCallback(),
    WandbCallback(
        gradient_save_freq=1000,
        model_save_path=f"models/{run.id}",
        verbose=0,
    )
]

print("Начинаем финальное обучение на Train выборке...")
model.learn(total_timesteps=2_000_000, progress_bar=True, callback=callbacks)

# Сохраняем модель и нормализатор!
model.save(model_widget.value)
vec_train.save("models/vec_normalize_prod_cnn_2m.pkl")
print("Модель сохранена.")

🚀 Initializing PPO (CNN) on device: mps


Output()

/Users/maximsinyaev/projects/trading_rl/trading_rl/.venv/lib/python3.12/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


Начинаем финальное обучение на Train выборке...


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.

Модель сохранена.


## 🔬 Валидация (Через Утилиту)
Этот блок использует нашу универсальную функцию валидации. При каждом запуске он выбирает случайный кусок графика (если случайный старт включен) и показывает результат торговли.


In [5]:

from utils.validation import run_validation

# Запускаем финальный продакшен бэктест на ПОСЛЕДНИХ 2 месяцах (без рандома!)
run_validation(
    data_features=data_features,
    model_path=model_widget.value,
    norm_path=norm_widget.value,
    use_frame_stack=True,
    n_stack=10,
    test_size=360,
    random_start=False
)


Loading model from models/ppo_prod_model...
Using the last test_size candles for validation.
Running simulation...

📊 VALIDATION REPORT
Initial balance: $100,000.00
Final balance:   $91,852.15
Net Profit:      -8.15%
Candles in pos:  196 out of 222
Status: 💀 Liquidated (Margin Call / Max Drawdown)
